In [3]:
# ==============================================================
# 🌍 Environmental Policy Compliance Assistant (RAG App)
# Gradio + Chroma + Embeddings + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------
!pip install gradio chromadb sentence-transformers pypdf transformers

# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline

# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
print("Loading embedding model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded")

# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------
client = chromadb.Client()

try:
    client.delete_collection("environmental_docs")
except:
    pass

collection = client.create_collection("environmental_docs")

# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------
print("Loading LLM...")
llm = pipeline("text-generation", model="google/flan-t5-base")
print("LLM loaded successfully")

# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF (Environmental Regulation)
# --------------------------------------------------------------
def process_pdf(file):
    print("Processing PDF...")
    reader = PdfReader(file.name)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    chunks = chunk_text(text)
    print("Total chunks:", len(chunks))
    for i, chunk in enumerate(chunks):
        embedding = embedding_model.encode(chunk).tolist()
        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )
    return f"Environmental regulation processed successfully. {len(chunks)} chunks stored."

# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------
def retrieve(query, k=3):
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    docs = results["documents"][0]
    print("\nRetrieved Chunks:\n", docs)
    return docs

# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------
def answer_question(query):
    docs = retrieve(query)
    context = " ".join(docs)
    prompt = f"""
You are an Environmental Policy Compliance Assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short, clear, and actionable compliance answer for engineers, sustainability officers, and executives.
"""
    response = llm(prompt, max_length=200, temperature=0.2)
    result = response[0]["generated_text"]
    print("\nRaw Model Output:\n", result)
    return result

# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------
def chat(question):
    if not question.strip():
        return "Please enter a question."
    answer = answer_question(question)
    if not answer:
        return "No answer generated."
    return answer

# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------
with gr.Blocks() as demo:
    gr.Markdown("# 🌍 Environmental Policy Compliance Assistant (RAG App)")

    gr.Markdown("""
Upload an environmental regulation PDF and ask questions about:

• emission limits
• waste management obligations
• renewable energy targets
• penalties for violations
• compliance and enforcement
""")

    pdf_input = gr.File(label="Upload Environmental Regulation PDF")
    upload_button = gr.Button("Process Regulation")
    status = gr.Textbox(label="Status")

    upload_button.click(process_pdf, inputs=pdf_input, outputs=status)

    question_box = gr.Textbox(label="Ask a Compliance Question")
    answer_box = gr.Textbox(label="Answer", lines=15)
    ask_button = gr.Button("Ask")

    ask_button.click(chat, inputs=question_box, outputs=answer_box)

# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------
demo.launch()

  Using cached chromadb-1.5.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached pypdf-6.9.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.24.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://746f9a74e4e47ae9f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
